### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [1]:
%uv pip install numpy scikit-learn


/home/lvillal/CEIA/ceia_uv/.venv/bin/python: No module named uv
Note: you may need to restart the kernel to use updated packages.


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [3]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [4]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [5]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [6]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [7]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [8]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [9]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [10]:
tfidfvect.vocabulary_['cocoliso']

KeyError: 'cocoliso'

Es muy útil tener el diccionario opuesto que va de índices a términos

In [11]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [12]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [13]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [14]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [15]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [16]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [17]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 9019, 9016, 8748], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [18]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [19]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [20]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [21]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [22]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [23]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


### 1. Vectorizar documentos y estudiar similaridad


Se toma 5 documentos al azar del conjunto de train y, para cada uno, se calcula la similaridad coseno contra el resto de los documentos de train usando la matriz TF-IDF X_train ya calculada. Se selcciona 5 documentos más similares a cada uno (excluyendo al propio documento) y se compara sus etiquetas.

In [40]:
rng = np.random.default_rng(42)
random_idxs = rng.choice(X_train.shape[0], size=5, replace=False)
print(f'Documentos elegidos al azar (índices): {random_idxs}')


Documentos elegidos al azar (índices): [8754 4965 7404 1009 4899]


In [41]:
def top_k_similares(idx, X, k=5):
    # Devuelve los índices de los k documentos más similares a X[idx] (sin contar el propio documento)
    cossim = cosine_similarity(X[idx], X)[0]
    orden = np.argsort(cossim)[::-1]
    orden = orden[orden != idx]  # sacamos el propio documento
    return orden[:k], cossim[orden[:k]]


for idx in random_idxs:
    print('------------')
    print(f'DOCUMENTO idx={idx}. clase: {newsgroups_train.target_names[y_train[idx]]}')
    print('---')
    print(newsgroups_train.data[idx][:400], '...\n')
    print('---')

    similares, scores = top_k_similares(idx, X_train, k=5)
    misma_clase = 0
    for pos, (sim_idx, score) in enumerate(zip(similares, scores), start=1):
        clase_sim = newsgroups_train.target_names[y_train[sim_idx]]
        coincide = clase_sim == newsgroups_train.target_names[y_train[idx]]
        misma_clase += coincide
        print(f'{pos}) idx={sim_idx} | similaridad={score:.3f} | clase={clase_sim} '
              f'{"ok misma clase" if coincide else ""}')
        print(f'     "{newsgroups_train.data[sim_idx][:150].strip()}..."')
    print(f'\n--> {misma_clase}/5 de los documentos más similares comparten la clase del documento origen')


------------
DOCUMENTO idx=8754. clase: talk.religion.misc
---

/(hudson)
/If someone inflicts pain on themselves, whether they enjoy it or not, they
/are hurting themselves.  They may be permanently damaging their body.

That is true.  It is also none of your business.  

Some people may also reason that by reading the bible and being a Xtian
you are permanently damaging your brain.  By your logic, it would be OK
for them to come into your home, take away yo ...

---
1) idx=6552 | similaridad=0.490 | clase=talk.religion.misc ok misma clase
     "If I have a habit that I really want to break, and I am willing to
make whatever sacrifice I need to make to break it, then I do so.
There have been..."
2) idx=10613 | similaridad=0.481 | clase=talk.religion.misc ok misma clase
     "/(hudson)
/Yes you do.  Who is to say that it is immoral for onesself to experience
/pain or to be hurt in some other way.  Maybe unpleasant, but that..."
3) idx=3616 | similaridad=0.465 | clase=talk.religion.misc

**Interpretación:**

Con la semilla fijada se sortearon los documentos [8754, 4965, 7404, 1009, 4899]. Los resultados de coincidencia de clase entre cada documento y sus 5 vecinos más similares fueron:

| idx  | clase real                | coincidencias (de 5) | similaridad top-1 |
|------|----------------------------|:---:|:---:|
| 8754 | `talk.religion.misc`       | 4/5 | 0.490 |
| 4965 | `comp.sys.mac.hardware`    | 2/5 | 0.365 |
| 7404 | `comp.os.ms-windows.misc`  | 0/5 | 0.268 |
| 1009 | `talk.politics.guns`       | 4/5 | 0.147 |
| 4899 | `sci.crypt`                | 3/5 | 0.197 |

- **Los mejores resultados (8754 y 1009, 4/5)** corresponden a documentos con vocabulario religioso/moral en un caso, argumentación sobre armas en el otro. Al tener términos poco frecuentes en el resto del corpus, esos términos concentran mucho peso TF-IDF y el coseno logra recuperar documentos del mismo subforo de discusión.
- **El caso 7404 (0/5)** : el documento es muy corto y genérico ("¿Es posible minimizar el administrador de programas al iniciar una aplicación...?"), sin vocabulario distintivo. Sus 5 vecinos más similares tienen similaridades bajas (0.22–0.27) y provienen de clases muy relacionadas pero no idénticas: `comp.windows.x`, `comp.graphics`, `sci.med`. Esto no es realmente un error: **20 Newsgroups tiene categorías con fronteras temáticas muy finas dentro de "computación"** (`comp.os.ms-windows.misc`, `comp.windows.x`, `comp.sys.ibm.pc.hardware`, `comp.sys.mac.hardware`, `comp.graphics`), por lo que un documento genérico sobre software puede "caer" en cualquiera de ellas sin que el contenido esté realmente desalineado.
- El caso 4965 (`comp.sys.mac.hardware`, 2/5) muestra el mismo fenómeno: sus vecinos "equivocados" pertenecen a `comp.sys.ibm.pc.hardware` y `comp.graphics`, es decir, hablan de puertos de impresora y hardware en general — semánticamente cercanos aunque la etiqueta no coincida.
- El caso 4899 (`sci.crypt`, 3/5) mezcla temas de privacidad/tecnología con política (`talk.politics.mideast`, `alt.atheism`), lo cual también tiene sentido de contenido: el propio documento habla de control de información y medios, un tema fronterizo entre criptografía/privacidad y política.
- En general, las similaridades obtenidas son **bajas en valor absoluto** (0.14 a 0.49): al remover headers/footers/quotes, muchos posts quedan muy cortos, lo que reduce el solapamiento léxico posible entre documentos aun siendo del mismo tema.
- **Conclusión**: la similaridad coseno sobre TF-IDF funciona razonablemente bien para recuperar documentos del mismo tema cuando el documento de consulta tiene vocabulario específico, pero es sensible tanto a documentos cortos/genéricos como a la granularidad fina de las categorías del dataset (varias clases distintas cubren temas solapados).

---
### 2. Modelo de clasificación por prototipos (zero-shot)


Se construye un clasificador sin entrenamiento ("zero-shot"): para cada documento de test se calcula la similaridad coseno contra todos los documentos de train, y se asigna la clase del documento de train con mayor similaridad (el "vecino más cercano", 1-NN, usando similaridad coseno sobre TF-IDF).

Como `X_test` tiene 7532 documentos y `X_train` tiene 11314, se calcula la matriz de similaridad por lotes (*batches*) para no consumir demasiada memoria.

In [42]:
def clasificador_por_prototipos(X_test, X_train, y_train, batch_size=500):
    n_test = X_test.shape[0]
    y_pred = np.empty(n_test, dtype=y_train.dtype)

    for start in range(0, n_test, batch_size):
        end = min(start + batch_size, n_test)
        sims = cosine_similarity(X_test[start:end], X_train)   # (batch, n_train)
        vecino_mas_cercano = np.argmax(sims, axis=1)
        y_pred[start:end] = y_train[vecino_mas_cercano]

    return y_pred


y_pred_zeroshot = clasificador_por_prototipos(X_test, X_train, y_train)
f1_zeroshot = f1_score(y_test, y_pred_zeroshot, average='macro')
print(f'F1-score macro (clasificador por prototipos / zero-shot): {f1_zeroshot:.4f}')
print(f'F1-score macro (Naïve Bayes entrenado, referencia de la celda anterior): {f1_score(y_test, y_pred, average="macro"):.4f}')


F1-score macro (clasificador por prototipos / zero-shot): 0.5050
F1-score macro (Naïve Bayes entrenado, referencia de la celda anterior): 0.5854


**Interpretación:**

- El clasificador por prototipos (1-NN por similaridad coseno) obtuvo F1-macro = 0.5050,  Naïve Bayes entrenado antes (con el vectorizador TF-IDF por defecto) diò F1-macro = 0.5854. Entonces, Naïve Bayes supera al clasificador zero-shot por casi 8%.
- El 1-NN decide la clase de un documento de test mirando a un único documento de train (el más parecido), lo cual lo hace muy vulnerable a que ese vecino sea un documento atípico, corto, o perteneciente a una categoría "vecina" semánticamente (como se vio en la Parte 1 con los ejemplos de las categorías comp.*). Naïve Bayes, agrega estadísticas de todos los documentos de cada clase (frecuencias de palabras condicionadas a la clase), lo que produce una decisión más estable y menos sensible a un solo documento ruidoso.
- El F1-macro = 0.50 para un método que no requiere ningún entrenamiento (ni siquiera ajustar una distribución de probabilidad) es un resultado que està bien como baseline: confirma que el espacio TF-IDF por sí solo ya captura información temática útil, aunque de forma más ruidosa que un clasificador entrenado.

---
### 3. Optimización de modelos Naïve Bayes


Se prueba distintas combinaciones de:
- **Parámetros del vectorizador**: max_df, min_df, sublinear_tf, stop_words, use_idf (sin tocar ngram_range, que queda en su valor por defecto (1, 1)).
- **Modelo Naïve Bayes**: MultinomialNB vs ComplementNB, variando el parámetro de suavizado alpha.

Para cada combinación se entrena sobre train y medimos F1-macro sobre test, y al final se elige con la mejor configuración.

In [43]:
resultados = []

vectorizer_configs = [
    {'max_df': 1.0, 'min_df': 1, 'sublinear_tf': False, 'stop_words': None},
    {'max_df': 1.0, 'min_df': 1, 'sublinear_tf': False, 'stop_words': 'english'},
    {'max_df': 0.9, 'min_df': 2, 'sublinear_tf': True,  'stop_words': 'english'},
    {'max_df': 0.7, 'min_df': 3, 'sublinear_tf': True,  'stop_words': 'english'},
    {'max_df': 0.5, 'min_df': 5, 'sublinear_tf': True,  'stop_words': 'english'},
]

modelos = {
    'MultinomialNB': MultinomialNB,
    'ComplementNB': ComplementNB,
}
alphas = [0.01, 0.1, 0.5, 1.0]

for vconf in vectorizer_configs:
    vect = TfidfVectorizer(ngram_range=(1, 1), **vconf)   # ngram_range fijo, NO se modifica
    Xtr = vect.fit_transform(newsgroups_train.data)
    Xte = vect.transform(newsgroups_test.data)

    for nombre_modelo, ModeloClase in modelos.items():
        for alpha in alphas:
            modelo = ModeloClase(alpha=alpha)
            modelo.fit(Xtr, y_train)
            preds = modelo.predict(Xte)
            f1 = f1_score(y_test, preds, average='macro')
            resultados.append({
                'modelo': nombre_modelo,
                'alpha': alpha,
                **vconf,
                'vocab_size': Xtr.shape[1],
                'f1_macro': f1,
            })

resultados = sorted(resultados, key=lambda r: r['f1_macro'], reverse=True)
print(f'{"modelo":14s} {"alpha":6s} {"max_df":7s} {"min_df":7s} {"sublin":7s} {"stopw":9s} {"vocab":7s}  f1_macro')
for r in resultados[:10]:
    print(f'{r["modelo"]:14s} {r["alpha"]:<6} {r["max_df"]:<7} {r["min_df"]:<7} '
          f'{str(r["sublinear_tf"]):7s} {str(r["stop_words"]):9s} {r["vocab_size"]:<7} {r["f1_macro"]:.4f}')

mejor = resultados[0]
print('\nMejor configuración encontrada:')
print(mejor)


modelo         alpha  max_df  min_df  sublin  stopw     vocab    f1_macro
ComplementNB   0.5    1.0     1       False   english   101322  0.6978
ComplementNB   0.5    1.0     1       False   None      101631  0.6961
ComplementNB   0.1    1.0     1       False   None      101631  0.6954
ComplementNB   0.5    0.9     2       True    english   39115   0.6951
ComplementNB   1.0    1.0     1       False   english   101322  0.6936
ComplementNB   1.0    1.0     1       False   None      101631  0.6930
ComplementNB   1.0    0.9     2       True    english   39115   0.6921
ComplementNB   0.1    1.0     1       False   english   101322  0.6919
ComplementNB   1.0    0.7     3       True    english   26747   0.6907
ComplementNB   0.5    0.7     3       True    english   26747   0.6903

Mejor configuración encontrada:
{'modelo': 'ComplementNB', 'alpha': 0.5, 'max_df': 1.0, 'min_df': 1, 'sublinear_tf': False, 'stop_words': 'english', 'vocab_size': 101322, 'f1_macro': 0.6978053768076979}


**Interpretación:**

La mejor configuración encontrada fue:


modelo: ComplementNB
alpha: 0.5
max_df: 1.0, min_df: 1  (sin filtrar vocabulario)
sublinear_tf: False
stop_words: 'english'
vocab_size: 101322
F1-macro: 0.6978


- **ComplementNB domina el top 10 completo**: no aparece ninguna configuración de MultinomialNB entre las mejores. ComplementNB fue diseñado para compensar el desbalance de clases usando estadísticas del *complemento* de cada clase, y en un dataset con 20 clases (con tamaños no perfectamente iguales) esto le da una ventaja consistente sobre MultinomialNB.
- **Mejora sustancial respecto al Naïve Bayes "de referencia"** de la sección anterior del notebook (MultinomialNB + TfidfVectorizer por defecto, F1 = 0.5854): la mejor configuración alcanza **0.6978**, una mejora de **+11.2 puntos** de F1-macro. La ganancia viene de dos factores combinados: cambiar de MultinomialNB a ComplementNB, y ajustar alpha (el suavizado por defecto, alpha=1.0, es subóptimo; valores intermedios como 0.5 funcionan mejor).
- **stop_words='english' ayuda consistentemente** (compara la fila 1 vs la fila 2 del ranking: mismo alpha, mismos max_df/min_df, la única diferencia es quitar stopwords, y el F1 sube de 0.6961 a 0.6978), ya que elimina palabras funcionales sin contenido temático que solo agregan ruido.
- Sorprendentemente, **restringir el vocabulario con min_df/max_df no mejora el resultado en este caso** (compara la fila 1, vocabulario completo de ~101k términos, con la fila 4, vocabulario reducido a ~39k términos con sublinear_tf=True): el F1 baja levemente (0.6978 → 0.6951). Esto sugiere que, para Naïve Bayes con este dataset, los términos poco frecuentes sí aportan señal discriminativa entre las 20 clases y conviene no descartarlos — aunque la pérdida es marginal (0.003), por lo que si el objetivo fuera reducir el costo computacional, esa configuración reducida seguiría siendo una alternativa muy razonable (más de 2.5x menos dimensiones por una caída de F1 despreciable).
- sublinear_tf=True (atenuar el TF con logaritmo) no aparece en la configuración ganadora, pero sí en varias de las siguientes mejores combinadas con vocabularios más chicos, lo que indica que su efecto es positivo principalmente quando se recorta agresivamente el vocabulario.

---
### 4. Transponer la matriz documento-término y estudiar similaridad entre palabras


Trasponemos X_train (documentos × términos) para obtener una matriz **término-documento** (términos × documentos). Cada fila de esta matriz traspuesta es entonces un vector que representa a una palabra en función de los documentos en los que aparece (y con qué peso TF-IDF); es decir, obtenemos una "vectorización de palabras" muy simple.

Elegimos manualmente 5 palabras interpretables y representativas de distintos tópicos del dataset (automóviles, religión, informática, deportes y espacio) y calculamos, para cada una, sus 5 palabras más similares según el coseno entre estos vectores.

In [44]:
X_train_T = X_train.T  # matriz término-documento: (vocab_size, n_documentos)
print(f'shape de la matriz término-documento: {X_train_T.shape}')

# Palabras elegidas manualmente por ser interpretables y representativas de distintos temas del dataset
palabras_elegidas = ['car', 'god', 'computer', 'hockey', 'space']

for palabra in palabras_elegidas:
    idx_palabra = tfidfvect.vocabulary_[palabra]
    sims = cosine_similarity(X_train_T[idx_palabra], X_train_T)[0]
    orden = np.argsort(sims)[::-1]
    orden = orden[orden != idx_palabra][:5]
    similares = [(idx2word[i], round(sims[i], 3)) for i in orden]
    print(f'{palabra:12s} -> {similares}')


shape de la matriz término-documento: (101631, 11314)
car          -> [('cars', np.float64(0.18)), ('criterium', np.float64(0.177)), ('civic', np.float64(0.175)), ('owner', np.float64(0.169)), ('dealer', np.float64(0.168))]
god          -> [('jesus', np.float64(0.269)), ('bible', np.float64(0.262)), ('that', np.float64(0.256)), ('existence', np.float64(0.255)), ('christ', np.float64(0.251))]
computer     -> [('decwriter', np.float64(0.156)), ('deluged', np.float64(0.152)), ('harkens', np.float64(0.152)), ('shopper', np.float64(0.144)), ('the', np.float64(0.136))]
hockey       -> [('ncaa', np.float64(0.274)), ('nhl', np.float64(0.265)), ('affiliates', np.float64(0.248)), ('xenophobes', np.float64(0.243)), ('sportschannel', np.float64(0.223))]
space        -> [('nasa', np.float64(0.33)), ('seds', np.float64(0.297)), ('shuttle', np.float64(0.293)), ('enfant', np.float64(0.28)), ('seti', np.float64(0.246))]


**Interpretación (en base a la ejecución real):**

Resultados obtenidos (palabra → 5 más similares con similaridad coseno):

| palabra | top-5 similares | calidad |
|---|---|---|
| `car` | `cars` (0.180), `criterium` (0.177), `civic` (0.175), `owner` (0.169), `dealer` (0.168) | buena |
| `god` | `jesus` (0.269), `bible` (0.262), `that` (0.256), `existence` (0.255), `christ` (0.251) | muy buena (con ruido) |
| `computer` | `decwriter` (0.156), `deluged` (0.152), `harkens` (0.152), `shopper` (0.144), `the` (0.136) | mala |
| `hockey` | `ncaa` (0.274), `nhl` (0.265), `affiliates` (0.248), `xenophobes` (0.243), `sportschannel` (0.223) | buena |
| `space` | `nasa` (0.330), `seds` (0.297), `shuttle` (0.293), `enfant` (0.280), `seti` (0.246) | muy buena |

- **`god`, `space` y `hockey` dan resultados muy interpretables**: `god` se asocia a `jesus`, `bible`, `christ` (vocabulario religioso); `space` a `nasa`, `shuttle`, `seti` y `seds` (Students for the Exploration and Development of Space, una organización real muy mencionada en `sci.space`); `hockey` a `ncaa`, `nhl` y `sportschannel` (competencias y canales deportivos). Esto confirma que la co-ocurrencia a nivel de documento en TF-IDF traspuesto sí logra capturar campos semánticos coherentes.
- **`car` da un resultado mayormente bueno** (`cars`, `civic` —un modelo de auto—, `owner`, `dealer`), salvo por `criterium` (un tipo de carrera ciclística), que aparece por coincidir casualmente con `car` en algún subconjunto de posts sobre carreras/autos de competición.
- **`computer` es el peor caso**: sus vecinos (`decwriter`, `deluged`, `harkens`, `shopper`, `the`) casi no tienen relación semántica clara, y las similaridades son las más bajas de las cinco palabras (0.13–0.16). La explicación más probable es que `computer` es un término **demasiado genérico y transversal**: aparece repartido de forma pareja en documentos de `comp.sys.mac.hardware`, `comp.sys.ibm.pc.hardware`, `comp.windows.x`, `comp.graphics`, etc., por lo que no tiene un patrón de co-ocurrencia distintivo con ningún grupo reducido de palabras — se "diluye" entre muchos subtemas de informática en lugar de identificarse con uno solo.
- También se observa la aparición de **stopwords sin filtrar** (`that` junto a `god`, `the` junto a `computer`): como en esta parte se reutilizó el vectorizador `tfidfvect` original (sin `stop_words='english'`), palabras funcionales de alta frecuencia terminan aaproximándose por pura co-ocurrencia masiva en casi todos los documentos. Esto sugiere que, para un análisis de similaridad de palabras más limpio, convendría usar un vectorizador con `stop_words='english'` (como el que se encontró óptimo en la Parte 3).
- **Conclusión**: el método capta bien palabras de contenido específico y poco ambiguas dentro del corpus (nombres propios, siglas, términos técnicos acotados a un tema), pero falla con palabras muy frecuentes y transversales a varias categorías, y arrastra ruido de stopwords si el vectorizador no las filtra explícitamente.